In [1]:
import pandas as pd
import numpy as np
import sys

sys.path.append("../scripts")   # path from notebook → per10.py folder
sys.path.append("../processed")

from model_update import (
    compute_weighted_trait_score,
    bayesian_update,
    run_bayesian_player_updates
)

from per10 import (
    compute_all_per10_360,
    compute_all_anticipation,
    compute_all_separation,
    compute_all_execution,
    compute_all_eyes,
    compute_all_innovation,
    compute_all_improv
)

In [2]:
df = pd.read_csv("../processed/input_cleaned_w1_2_3.csv")
df.head()

,game_id,play_id,nfl_id,frame_id,player_position,player_side,player_role,x,y,s,...,season,week,pass_result,pass_length,route_of_targeted_receiver,team_coverage_man_zone,yards_gained,expected_points,expected_points_added,throw_frame
0,2023090700,101,52546,16,CB,Defense,Defensive Coverage,46.73,12.92,3.17,...,2023,1,I,22,CORNER,ZONE_COVERAGE,0,0.927021,-2.145443,26
1,2023090700,101,52546,17,CB,Defense,Defensive Coverage,47.02,13.03,3.09,...,2023,1,I,22,CORNER,ZONE_COVERAGE,0,0.927021,-2.145443,26
2,2023090700,101,52546,18,CB,Defense,Defensive Coverage,47.27,13.12,2.77,...,2023,1,I,22,CORNER,ZONE_COVERAGE,0,0.927021,-2.145443,26
3,2023090700,101,52546,19,CB,Defense,Defensive Coverage,47.51,13.19,2.39,...,2023,1,I,22,CORNER,ZONE_COVERAGE,0,0.927021,-2.145443,26
4,2023090700,101,52546,20,CB,Defense,Defensive Coverage,47.73,13.23,2.03,...,2023,1,I,22,CORNER,ZONE_COVERAGE,0,0.927021,-2.145443,26


In [3]:
sample_play_ids = df["play_id"].drop_duplicates().sample(n=15, random_state=42)
sample_df = df[df["play_id"].isin(sample_play_ids)]

In [4]:
per10_df = compute_all_per10_360(compute_all_anticipation(sample_df), compute_all_separation(sample_df), 
                                 compute_all_execution(sample_df), compute_all_eyes(sample_df),
                                 compute_all_innovation(sample_df), compute_all_improv(sample_df))

per10_df.head()

,play_id,nfl_id,A,S,E,Eyes,Innovation,Improv,PER10_360
0,147,47859,4,3.0,6,4.87,0.42,2.14,3
1,147,53511,10,8.0,7,4.94,0.04,2.64,5
2,147,53565,6,3.0,5,6.08,0.49,2.67,4
3,147,54481,6,3.0,6,3.27,0.51,2.26,4
4,147,56045,4,3.0,7,3.86,0.31,2.95,4


In [5]:
#we merge to get the player side column information which we will need for the weighted trait

per10_df = per10_df.merge(sample_df[['nfl_id', 'player_side', 'game_id']], on='nfl_id', how='left')

per10_df.head()


,play_id,nfl_id,A,S,E,Eyes,Innovation,Improv,PER10_360,player_side,game_id
0,147,47859,4,3.0,6,4.87,0.42,2.14,3,Offense,2023091007
1,147,47859,4,3.0,6,4.87,0.42,2.14,3,Offense,2023091007
2,147,47859,4,3.0,6,4.87,0.42,2.14,3,Offense,2023091007
3,147,47859,4,3.0,6,4.87,0.42,2.14,3,Offense,2023091007
4,147,47859,4,3.0,6,4.87,0.42,2.14,3,Offense,2023091007


In [6]:
per10_df.columns

Index(['play_id', 'nfl_id', 'A', 'S', 'E', 'Eyes', 'Innovation', 'Improv',
       'PER10_360', 'player_side', 'game_id'],
      dtype='object')

In [7]:
per10_df["weighted_trait"] = per10_df.apply(compute_weighted_trait_score, axis=1)

per10_df.head(15)

,play_id,nfl_id,A,S,E,Eyes,Innovation,Improv,PER10_360,player_side,game_id,weighted_trait
0,147,47859,4,3.0,6,4.87,0.42,2.14,3,Offense,2023091007,3.715
1,147,47859,4,3.0,6,4.87,0.42,2.14,3,Offense,2023091007,3.715
2,147,47859,4,3.0,6,4.87,0.42,2.14,3,Offense,2023091007,3.715
3,147,47859,4,3.0,6,4.87,0.42,2.14,3,Offense,2023091007,3.715
4,147,47859,4,3.0,6,4.87,0.42,2.14,3,Offense,2023091007,3.715
5,147,47859,4,3.0,6,4.87,0.42,2.14,3,Offense,2023091007,3.715
6,147,47859,4,3.0,6,4.87,0.42,2.14,3,Offense,2023091007,3.715
7,147,47859,4,3.0,6,4.87,0.42,2.14,3,Offense,2023091007,3.715
8,147,47859,4,3.0,6,4.87,0.42,2.14,3,Offense,2023091007,3.715
9,147,47859,4,3.0,6,4.87,0.42,2.14,3,Offense,2023091007,3.715


In [8]:
bayesian_df = run_bayesian_player_updates(per10_df)
bayesian_df.head()

,nfl_id,game_id,play_id,bayesian_rating,bayesian_uncertainty
0,37078,2023091006,1096,4.870,0.800
1,37078,2023091006,1096,4.856,0.444
2,37078,2023091006,1096,4.850,0.308
3,37078,2023091006,1096,4.848,0.235
4,37078,2023091006,1096,4.846,0.190


In [9]:
bayesian_df.groupby("nfl_id")["bayesian_rating"].last().sort_values(ascending=False)

nfl_id
54475    7.882
42441    7.798
44906    7.398
43336    7.323
52272    7.247
         ...  
53565    4.336
43522    4.292
54481    4.099
56045    3.931
47859    3.744
Name: bayesian_rating, Length: 71, dtype: float64